# Follow the Line: Explore

Our robot has a camera pointing **down** at the pool floor. There's an orange-red line of tape on the
floor, and the robot needs to follow it. Your job: look at a camera image and decide whether the robot
should go **LEFT**, **RIGHT**, **STRAIGHT**, or say it's **LOST** (no line in view).

Here's the plan, one section per step:

1. Load an image
2. Understand color spaces (why we use HSV)
3. Make a **mask**: white where the line is, black everywhere else
4. Clean up the mask
5. Find the blobs in the mask and pick the one that's the line
6. Find where the line is and decide which way to steer
7. Test on every image

Run a cell with **Shift + Enter**. Cells marked **🔧 TODO** need you to change something.

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

IMAGES = Path("../images")
ANSWERS = json.loads((IMAGES / "answers.json").read_text())


def show(*images, titles=None, size=5):
    """Shows images side by side. Color images are expected in OpenCV's BGR order."""
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size * 0.75))
    for i, ax in enumerate(np.atleast_1d(axes)):
        image = images[i]
        if image.ndim == 3:
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(image, cmap="gray")
        ax.set_title(titles[i] if titles else "")
        ax.axis("off")
    plt.show()

## 1. Load an image

To a computer, an image is just a grid of numbers. OpenCV loads it as a NumPy array with the shape
**(height, width, 3)**. Each pixel has 3 numbers for its color, from 0 to 255.

In [ ]:
img = cv2.imread(str(IMAGES / "01_center.png"))
print("shape (height, width, channels):", img.shape)
print("top-left pixel:", img[0, 0])

plt.imshow(img)
plt.axis("off")
plt.show()

The line should be orange, but it's blue! OpenCV stores the 3 color numbers in the order
**Blue, Green, Red (BGR)**, but matplotlib expects **Red, Green, Blue (RGB)**. So red and blue got swapped.
The `show()` helper converts BGR to RGB for you:

In [ ]:
show(img)

## 2. Color spaces: why HSV?

In BGR, "orange" depends on all three numbers at once, and if the water gets darker, all three change.
That makes it hard to write a rule like "keep the orange pixels."

**HSV** splits color up differently:

| Channel | Meaning | Range in OpenCV |
|---|---|---|
| **H**ue | *Which* color: red, yellow, green, blue... | 0–179 |
| **S**aturation | How vivid it is (0 = gray) | 0–255 |
| **V**alue | How bright it is (0 = black) | 0–255 |

Now "orange" is mostly one number (hue), no matter how bright the water is.

In [ ]:
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(hsv)
show(h, s, v, titles=["Hue", "Saturation", "Value"], size=4)

Let's measure some pixels. Arrays are indexed `[row, column]`, which is `[y, x]`: y counts **down** from the top.

In [ ]:
samples = {
    "line":          ("01_center.png", 240, 320),
    "water":         ("01_center.png", 240, 100),
    "line (murky)":  ("07_murky_left.png", 240, 190),
    "yellow buoy":   ("09_seaweed_and_buoy.png", 150, 470),
    "green seaweed": ("09_seaweed_and_buoy.png", 380, 159),
    "orange fish":   ("10_fish_only.png", 260, 420),
}
for label, (name, y, x) in samples.items():
    pixel = cv2.cvtColor(cv2.imread(str(IMAGES / name)), cv2.COLOR_BGR2HSV)[y, x]
    print(f"{label:14s} H={pixel[0]:3d}  S={pixel[1]:3d}  V={pixel[2]:3d}")

**Think about it:** which numbers separate the line from the water, the buoy, and the seaweed?
How dark does the line get in murky water?

Notice the fish is almost the **same color** as the line. Color alone can't tell them apart.
We'll deal with that in step 5.

## 3. Make a mask 🔧 TODO

`cv2.inRange` keeps every pixel whose H, S, and V are all between `LOWER` and `UPPER`.
Kept pixels become white (255), everything else black (0).

Right now the range keeps **every** pixel, so the mask is all white. Using your measurements above,
change `LOWER` and `UPPER` so the mask only keeps the line. Check it on both test images below.

In [ ]:
# 🔧 TODO: pick a range that keeps the line and nothing else.
LOWER = np.array([0, 0, 0])        # lowest  [H, S, V] to keep
UPPER = np.array([179, 255, 255])  # highest [H, S, V] to keep


def make_mask(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    return cv2.inRange(hsv, LOWER, UPPER)


for name in ["09_seaweed_and_buoy.png", "07_murky_left.png"]:
    test = cv2.imread(str(IMAGES / name))
    show(test, make_mask(test), titles=[name, "mask"], size=4)

## 4. Clean up the mask

Real masks have little specks of noise and small holes. **Morphology** fixes that by sliding a small
shape (the *kernel*) over the mask:

- **Opening** (shrink, then grow back) erases specks smaller than the kernel.
- **Closing** (grow, then shrink back) fills holes smaller than the kernel.

In [ ]:
KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))


def clean(mask):
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, KERNEL)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, KERNEL)
    return mask


raw = make_mask(cv2.imread(str(IMAGES / "07_murky_left.png")))
show(raw, clean(raw), titles=["before", "after"], size=4)

## 5. Find the blobs and pick the line 🔧 TODO

`cv2.findContours` traces the outline (the *contour*) of every white blob in the mask.
For each blob we measure two things:

- **Area:** how many pixels it covers.
- **Aspect ratio:** long side ÷ short side. A line is long and thin (big aspect ratio).
  A fish or a buoy is chubby (small aspect ratio).

In [ ]:
def describe(contour):
    area = cv2.contourArea(contour)
    (_, _), (w, h), _ = cv2.minAreaRect(contour)  # the smallest rectangle around the blob
    aspect = max(w, h) / max(min(w, h), 1)
    return area, aspect


for name in ["08_fish_and_line.png", "11_short_line_and_big_buoy.png"]:
    test = cv2.imread(str(IMAGES / name))
    contours, _ = cv2.findContours(clean(make_mask(test)), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    drawn = test.copy()
    print(name)
    for i, contour in enumerate(contours):
        area, aspect = describe(contour)
        print(f"  blob {i}: area = {area:7.0f} pixels, aspect ratio = {aspect:5.1f}")
        cv2.drawContours(drawn, [contour], -1, (0, 255, 0), 3)
        x, y, _, _ = cv2.boundingRect(contour)
        cv2.putText(drawn, str(i), (x + 5, y + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    show(drawn, size=4)

Pick limits that keep the line but throw away fish, buoys, and specks.

In [ ]:
# 🔧 TODO: pick limits that only let the line through.
MIN_AREA = 0      # ignore blobs smaller than this many pixels
MIN_ASPECT = 0.0  # ignore blobs less stretched-out than this


def pick_line(mask):
    """Returns the contour of the line, or None if there isn't one."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lines = []
    for contour in contours:
        area, aspect = describe(contour)
        if area >= MIN_AREA and aspect >= MIN_ASPECT:
            lines.append(contour)
    if not lines:
        return None
    return max(lines, key=cv2.contourArea)

## 6. Where is the line? 🔧 TODO

To steer, we need the line's center. `cv2.moments(contour)` returns a dictionary of sums over the blob's pixels:

- `m["m00"]` is the blob's area.
- `m["m10"]` is the sum of every pixel's x coordinate.

So the **average x** (the center) is `m["m10"] / m["m00"]`.

Then turn it into an **offset** from -1.0 (far left) to 0.0 (center) to 1.0 (far right).
For a 640-pixel-wide image: x = 0 → -1.0, x = 320 → 0.0, x = 640 → 1.0.

Fill in `offset_of` below.

In [ ]:
DEADBAND = 0.15  # within 15% of the center counts as straight


def offset_of(contour, width):
    """-1.0 = far left, 0.0 = center, 1.0 = far right."""
    # 🔧 TODO: use cv2.moments to find the center x, then turn it into an offset.
    return 0.0


def decide(image):
    line = pick_line(clean(make_mask(image)))
    if line is None:
        return "LOST"
    offset = offset_of(line, image.shape[1])
    if offset < -DEADBAND:
        return "LEFT"
    if offset > DEADBAND:
        return "RIGHT"
    return "STRAIGHT"


print(decide(cv2.imread(str(IMAGES / "02_left.png"))), "(should be LEFT)")

## 7. Test on every image

Green titles are right, red titles are wrong. Keep tuning until you get them all!

In [ ]:
names = sorted(ANSWERS)
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
score = 0
for ax, name in zip(axes.flat, names):
    image = cv2.imread(str(IMAGES / name))
    got, expected = decide(image), ANSWERS[name]["answer"]
    correct = got == expected
    score += correct
    line = pick_line(clean(make_mask(image)))
    if line is not None:
        image = cv2.drawContours(image.copy(), [line], -1, (0, 255, 0), 3)
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{name[:-4]}\ngot {got}, expected {expected}", color="green" if correct else "red", fontsize=10)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"Score: {score}/{len(names)}")

## 8. Move it into the codebase

A notebook is great for experimenting, but the robot runs regular Python files. Now:

1. Copy `solutions/_template.py` to `solutions/<your-github-username>.py`.
2. Move your code into its three functions:
   - `make_mask` + `clean` → `find_line_mask`
   - `pick_line` → `find_line`
   - `offset_of` → `line_offset`
3. In a terminal, run `pytest -k <your-github-username>` until everything passes.
4. Commit, push, and open a pull request. GitHub will run the same tests on your PR.

## Stretch goals

- **Look ahead.** `cv2.fitLine` gives the line's angle. Can you start turning *before* the line drifts
  off-center?
- **True red tape.** Real red wraps around the hue scale: it's near 0 **and** near 179. How would you
  keep both ends? (Hint: make two masks and combine them with `cv2.bitwise_or`.)
- **Make a harder test.** Add a new case to `tools/make_images.py` (a bent line, two lines, bubbles...)
  and see if your solution survives.